# Script thu thập 2 JWT token, trích `(r, s)` — System Demo 1 (JWT ES256)

**Mục tiêu:** gọi vào dịch vụ `system-demos/jwt-es256/app.py` (đang chạy thật, cố ý dễ tổn thương —
nonce `k` cố định) để lấy ≥2 JWT token, giải mã JWS compact serialization, trích `(header, payload, r, s)`
của từng token, và lưu lại thành bằng chứng cho bước khôi phục khóa (`recover_key.ipynb`, task kế tiếp).

**Yêu cầu trước khi chạy notebook này:** mở 1 terminal riêng, chạy server:
```bash
python3 system-demos/jwt-es256/app.py --port 8001
```
Notebook này **không** tự khởi động server (đúng kiến trúc 2 tiến trình: server ↔ client tấn công,
xem `require.md` mục 9 — sơ đồ phụ thuộc Track C).

**Input:** server đang chạy tại `http://127.0.0.1:8001` (đổi `BASE_URL` bên dưới nếu dùng cổng khác).
**Output:** `system-demos/jwt-es256/collected_tokens.json` — chứa public key server + danh sách token đã
thu thập kèm `(r, s)` đã trích sẵn, để `recover_key.ipynb` đọc lại mà không cần gọi HTTP lần nữa.
**Tham chiếu:** `docs/experimental_design.md` Track C (System Demo 1); `signatures/nonce_theory.md` §2–3
(vì sao cùng `k` ⇒ cùng `r`, bất kể thông điệp).
**Tái lập:** chạy headless được qua `jupyter nbconvert --to notebook --execute --inplace` **miễn là
server đã chạy sẵn** ở bước trên.

In [1]:
import requests
import base64
import json
import os

BASE_URL = "http://127.0.0.1:8001"

try:
    health = requests.get(f"{BASE_URL}/health", timeout=3).json()
    print("Server đang chạy:", health)
except requests.exceptions.ConnectionError as exc:
    raise RuntimeError(
        f"Không kết nối được {BASE_URL} — hãy chạy trước ở terminal khác:\n"
        f"  python3 system-demos/jwt-es256/app.py --port 8001"
    ) from exc

Server đang chạy: {'status': 'ok', 'alg': 'ES256', 'nonce_mode': 'FIXED_REUSED_NONCE_VULNERABLE', 'tokens_issued': 0}


## Phần 1 — Lấy khóa công khai của server (JWKS)

Cần để `recover_key.ipynb` xác nhận khóa khôi phục đúng (`d·G == Q`) mà không phải đoán.

In [2]:
jwks = requests.get(f"{BASE_URL}/jwks", timeout=3).json()
pubkey_info = jwks["keys"][0]
curve_params = jwks["curve_params"]

print("kid:", pubkey_info["kid"])
print("curve:", pubkey_info["crv"])
print("Qx:", curve_params["qx"])
print("Qy:", curve_params["qy"])
print("order_n:", curve_params["order_n"])

kid: jwt-es256-signing-key
curve: P-256
Qx: 23903884683187771954195029892783203457533510318083150669811299872665916295758
Qy: 114025755449830544111471691649295547177630684847043539135214914770243498951878
order_n: 115792089210356248762697446949407573529996955224135760342422259061068512044369


## Phần 2 — Thu thập ≥2 JWT token (2 người dùng khác nhau, 2 request `/auth/login` riêng biệt)

Server không yêu cầu mật khẩu (demo đơn giản hoá phần xác thực để tập trung vào lỗ hổng chữ ký) — chỉ cần
gửi `username` bất kỳ.

In [3]:
usernames = ["alice", "mallory"]  # 2 danh tính khác nhau -> 2 payload khác nhau
raw_tokens = []
for u in usernames:
    resp = requests.post(f"{BASE_URL}/auth/login", json={"username": u}, timeout=3)
    resp.raise_for_status()
    token = resp.json()["access_token"]
    raw_tokens.append((u, token))
    print(f"[{u}] token (rút gọn): {token[:40]}...{token[-20:]}")

[alice] token (rút gọn): eyJhbGciOiJFUzI1NiIsInR5cCI6IkpXVCIsImtp...hPOkCmR5xrBmaH1MmDPw
[mallory] token (rút gọn): eyJhbGciOiJFUzI1NiIsInR5cCI6IkpXVCIsImtp...klx0-IxjcZWLxoyOxGiQ


## Phần 3 — Giải mã JWS compact serialization, trích `(header, payload, r, s)`

JWT ES256 theo RFC 7518 §3.4: `signature = base64url(R || S)` — nối trực tiếp (KHÔNG phải DER), mỗi phần
`R`, `S` dài đúng `ceil(bitlen(n)/8)` byte (32 byte với P-256).

In [4]:
def b64url_decode(s: str) -> bytes:
    pad = "=" * (-len(s) % 4)
    return base64.urlsafe_b64decode(s + pad)


def parse_jwt(token: str) -> dict:
    header_b64, payload_b64, sig_b64 = token.split(".")
    header = json.loads(b64url_decode(header_b64))
    payload = json.loads(b64url_decode(payload_b64))
    sig_bytes = b64url_decode(sig_b64)
    n_bytes = len(sig_bytes) // 2  # R||S chia đôi đều nhau
    r = int.from_bytes(sig_bytes[:n_bytes], "big")
    s = int.from_bytes(sig_bytes[n_bytes:], "big")
    return {
        "header": header, "payload": payload,
        "r": str(r), "s": str(s),
        "signing_input": f"{header_b64}.{payload_b64}",
        "token": token,
    }


collected = [dict(username=u, **parse_jwt(tok)) for u, tok in raw_tokens]
for c in collected:
    print(f"[{c['username']}] payload={c['payload']}")
    print(f"           r={c['r']}")
    print(f"           s={c['s']}")

[alice] payload={'sub': 'alice', 'role': 'user', 'iat': 1795858376, 'exp': 1795861976, 'jti': 'tok-1-c88b2875'}
           r=45007991771160465399815027242781520863888777863911291601794406315176982294070
           s=33609144219449503782327094145276019749232079742913376264105841496394731979583
[mallory] payload={'sub': 'mallory', 'role': 'user', 'iat': 1795858376, 'exp': 1795861976, 'jti': 'tok-2-9293de8f'}
           r=45007991771160465399815027242781520863888777863911291601794406315176982294070
           s=109343230810456195637905988877342769965506115801196074801379572998871144220297


## Phần 4 — Đối chiếu `r` giữa các token (bằng chứng quan sát được trực tiếp)

Đây là bước đầu tiên của chuỗi tấn công: **chỉ cần nhìn `r`**, chưa cần biết gì về cơ chế nội bộ của
server, đã có thể nghi ngờ nonce bị tái sử dụng (đúng vai trò của `signatures/repeated_r_detector.py`,
sẽ dùng chính thức ở `recover_key.ipynb`).

In [5]:
r_values = {c["r"] for c in collected}
print(f"Số token thu được: {len(collected)}")
print(f"Số giá trị r PHÂN BIỆT: {len(r_values)}")
if len(r_values) < len(collected):
    print("=> PHÁT HIỆN TRÙNG r GIỮA CÁC TOKEN — dấu hiệu nonce k bị tái sử dụng.")
else:
    print("=> Không phát hiện trùng r trong lô token này (thử thu thập thêm).")

Số token thu được: 2
Số giá trị r PHÂN BIỆT: 1
=> PHÁT HIỆN TRÙNG r GIỮA CÁC TOKEN — dấu hiệu nonce k bị tái sử dụng.


## Phần 5 — Lưu bằng chứng ra file cho `recover_key.ipynb`

In [6]:
# Notebook nằm ngay trong system-demos/jwt-es256/ -> ghi file cùng thư mục
OUT_PATH = "collected_tokens.json"

output = {
    "server_base_url": BASE_URL,
    "pubkey": {
        "kid": pubkey_info["kid"], "curve": pubkey_info["crv"],
        "qx": curve_params["qx"], "qy": curve_params["qy"],
        "p": curve_params["p"], "a": curve_params["a"], "b": curve_params["b"],
        "gx": curve_params["gx"], "gy": curve_params["gy"], "order_n": curve_params["order_n"],
    },
    "tokens": collected,
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã lưu {len(collected)} token vào {OUT_PATH}")
print(f"r có trùng không: {len(r_values) < len(collected)}")

Đã lưu 2 token vào collected_tokens.json
r có trùng không: True


## Kết luận

- Thu thập thành công 2 JWT token thật từ server đang chạy, không cần biết trước bất kỳ nội tình nào
  của server (chỉ gọi 2 endpoint công khai: `/jwks`, `/auth/login`).
- Giải mã đúng JWS compact serialization theo RFC 7518 §3.4 (R||S, không phải DER).
- Phát hiện `r` trùng giữa 2 token — bằng chứng ban đầu, **chưa khôi phục khóa** (việc đó thuộc
  `recover_key.ipynb`, dùng `signatures/nonce_reuse_attacker.py` đọc lại `collected_tokens.json` vừa lưu).
- **Lưu ý khi dùng lại:** mỗi lần chạy server mới (không truyền `--seed`/`--secret-d`/`--fixed-k`), khóa
  và nonce sẽ **giống nhau** giữa các lần chạy vì `app.py` seed RNG mặc định `seed=2026` cố định — đây là
  đặc tính tái lập, không phải ngẫu nhiên thật; dùng `--secret-d`/`--fixed-k`/`--seed` khác nếu muốn đổi.